# BPE tokenizer

This is a companion notebook to the other ml_coding pactice notebooks in this folder.
Here we focus on implementing the BPE tokenizer.

In [1]:
# Imports
import regex as re

## Pretokenize

In [2]:
GPT2_PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def pretokenize(text: str, special_tokens: list[str] = None) -> dict[tuple[int, ...], int]:
    """
    Pretokenize text into a frequency map of byte-tuples.
    
    1. Split out special tokens so they don't get regex-tokenized
    2. Apply GPT-2 regex pattern to each non-special chunk
    3. Convert each match to a byte tuple and count frequencies
    """
    if not special_tokens:
        special_tokens = []

    # Build regex that matches special tokens OR the GPT-2 pattern
    if special_tokens:
        escaped = [re.escape(t) for t in special_tokens]
        pattern = re.compile(
            r"(?:{})|(?:{})".format("|".join(escaped), GPT2_PAT)
        )
    else:
        pattern = re.compile(GPT2_PAT)

    freq_map: dict[tuple[int, ...], int] = {}
    for m in pattern.finditer(text):
        tok = m.group(0)
        if tok in special_tokens:
            continue  # skip special tokens, don't include in BPE training
        key = tuple(tok.encode("utf-8"))
        freq_map[key] = freq_map.get(key, 0) + 1
    return freq_map


def pretokenize_file(
    filepath: str, special_tokens: list[str] = None
) -> dict[tuple[int, ...], int]:
    """Read a file and pretokenize it."""
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    return pretokenize(text, special_tokens)

## Simple BPE implementation

Here we assume that we start with the freq_map that we get after pretokenization above.

### Helper Functions

In [ ]:
def count_pairs(freq_map: dict[tuple[int, ...], int]) -> dict[tuple[int, int], int]:
    """Count all adjacent pairs across the frequency map."""
    pairs = {}
    for token_tuple, count in freq_map.items():
        for i in range(len(token_tuple) - 1):
            cur_pair = (token_tuple[i], token_tuple[i+1])
            pairs[cur_pair] = pairs.get(cur_pair, 0) + count
    return pairs

def get_best_pair(pair_counts: dict[tuple[int, int], int]):
    """Find the most frequent pair. Ties broken by lexicographically greater pair."""
    return max(pair_counts, key=lambda pair: (pair_counts[pair], pair))

def apply_merge(freq_map: dict[tuple[int, ...], int],
                best_pair: tuple[int, int],
                new_id: int):
    """Replace every occurrence of pair in freq_map keys with new_id."""
    new_freq_map = {}
    for token_tuple, count in freq_map.items():
        new_token_tuple = []
        i = 0

        while i < len(token_tuple):
            if i < len(token_tuple) - 1 and token_tuple[i] == best_pair[0] and token_tuple[i+1] == best_pair[1]:
                new_token_tuple.append(new_id)
                i += 2
            else:
                new_token_tuple.append(token_tuple[i])
                i += 1
        new_key = tuple(new_token_tuple)
        new_freq_map[new_key] = new_freq_map.get(new_key, 0) + count

    return new_freq_map

### Train BPE

In [15]:
def train_bpe(
        freq_map: dict[tuple[int, ...], int],
        vocab_size: int,
        special_tokens: list[str]
):
    """
    Train a BPE tokenizer.

    Args:
        freq_map: Pretokenized corpus as {byte_tuple: count}
        vocab_size: Target vocabulary size
        special_tokens: List of special token strings

    Returns:
        vocab: {token_id: bytes} for every token
        merges: Ordered list of (pair_a, pair_b) merge rules
        special_token_ids: {special_token_string: token_id}
    """
    # 1. Add the 256 bytes.
    vocab = {i: bytes([i]) for i in range(256)}

    # 2. Add special tokens.
    special_token_ids = {}
    for token in special_tokens:
        tid = len(vocab)
        vocab[tid] = token.encode("utf-8")
        special_token_ids[token] = tid

    # 3. Check how many merges can we have.
    num_merges = vocab_size - len(vocab)

    # 4. Run BPE training.
    merges = []
    for i in range(num_merges):
        pair_counts = count_pairs(freq_map)
        if not pair_counts:
            break               # Nothing left to merge so we can end.

        best_pair = get_best_pair(pair_counts)
        new_id = len(vocab)

        merges.append(best_pair)
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]

        freq_map = apply_merge(freq_map, best_pair, new_id)

        if i % 50 == 0:
            print(
                f"Merge {i}/{num_merges}: {best_pair} -> {new_id}  "
                f"(count={pair_counts[best_pair]}, "
                f"token='{vocab[new_id]}')"
            )

    print(f"Training completed.\nVocab size: {len(vocab)}\nMerges learned: {len(merges)}")
    return vocab, merges, special_token_ids

### Encoding

In [24]:
# Helper functions.

def _split_on_special_tokens(
    text: str, special_tokens: list[str]
) -> list[tuple[str, bool]]:
    """
    Split text into chunks, separating special tokens.
    Returns list of (chunk_text, is_special) tuples.
    """
    if not special_tokens:
        return [(text, False)]

    escaped = [re.escape(t) for t in special_tokens]
    pattern = re.compile(r"({})".format("|".join(escaped)))

    result = []
    for part in pattern.split(text):
        if not part:
            continue
        result.append((part, part in special_tokens))
    return result

def _apply_merges(
        ids: list[int],
        merges: list[tuple[int, int]],
        first_learned_id: int
    ) -> list[int]:
    """
    Repeatedly find the highest-priority (lowest-rank) merge present
    in ids and apply it everywhere, until no more merges apply.

    Args:
        ids: Starting token IDs (initially raw bytes 0-255)
        merges: Ordered merge rules from training
        first_new_id: Token ID of the first merge (256 + num_special_tokens)
    """
    # Build the lookup that we will use.
    merge_lookup = {}
    for i, (a, b) in enumerate(merges):
        merge_lookup[(a, b)] = (i, first_learned_id + i)

    while len(ids) >= 2:
        # Find the pair with the lowest rank before we apply the merge.
        best_pair = None
        best_rank = float("inf")

        for i in range(len(ids) - 1):
            cur_pair = (ids[i], ids[i+1])
            if cur_pair in merge_lookup:
                rank, _ = merge_lookup[cur_pair]
                if rank < best_rank:
                    best_rank = rank
                    best_pair = cur_pair

        if best_pair is None:
            break             # No applicable merges remain.

        # Apply the merge.
        _, new_id = merge_lookup[best_pair]
        new_ids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and best_pair[0] == ids[i] and best_pair[1] == ids[i+1]:
                new_ids.append(new_id)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        ids = new_ids

    return ids

In [25]:
# Main encoding function
GPT2_PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def encode(
    text: str,
    merges: list[tuple[int, int]],
    special_token_ids: dict[str, int],
) -> list[int]:
    """
    Encode text into token IDs using trained BPE merges.

    Args:
        text: Input string to encode
        merges: Ordered merge rules from training
        special_token_ids: {special_token_string: token_id}

    Returns:
        List of token IDs
    """
    special_tokens = list(special_token_ids.keys())
    first_learned_id = 256 + len(special_tokens)

    # 1: Split the text around special tokens and get the chunks.
    chunks = _split_on_special_tokens(text, special_tokens)

    # 2: Encode each chunk.
    gpt2_re = re.compile(GPT2_PAT)
    token_ids = []

    for chunk_text, is_special in chunks:
        if is_special:
            token_ids.append(special_token_ids[chunk_text])
        else:
            # Pretokenize and process each piece.
            for m in gpt2_re.finditer(chunk_text):
                ids = list(m.group(0).encode("utf-8"))
                ids = _apply_merges(ids, merges, first_learned_id)
                token_ids.extend(ids)

    return token_ids

### Decoding

In [27]:
def decode(token_ids: list[int], vocab: dict[int, bytes]) -> str:
    """Decode token IDs back to a string."""
    return b"".join(vocab[tid] for tid in token_ids).decode("utf-8")
    

## Test the functions.

In [8]:
freq_map = pretokenize("low low low lowest")
for byte_tuple, count in freq_map.items():
    print(f"'{bytes(byte_tuple).decode('utf-8')}' : {count}")

'low' : 1
' low' : 2
' lowest' : 1


In [7]:
freq_map = pretokenize("here's here's here's some text and I'd like to see how it is getting pretokenized with the regex that we have!")
for byte_tuple, count in freq_map.items():
    print(f"'{bytes(byte_tuple).decode('utf-8')}' : {count}")

'here' : 1
''s' : 3
' here' : 2
' some' : 1
' text' : 1
' and' : 1
' I' : 1
''d' : 1
' like' : 1
' to' : 1
' see' : 1
' how' : 1
' it' : 1
' is' : 1
' getting' : 1
' pretokenized' : 1
' with' : 1
' the' : 1
' regex' : 1
' that' : 1
' we' : 1
' have' : 1
'!' : 1


In [12]:
# One merge step, traced on the longer input above.
text = "here's here's here's some text and I'd like to see how it is getting pretokenized with the regex that we have!"
freq_map = pretokenize(text)

def show(b):
    """Render a byte value readably; merged ids (>= 256) show as <id>."""
    if b >= 256:
        return f"<{b}>"
    return repr(bytes([b]).decode("utf-8", errors="replace"))

pair_counts = count_pairs(freq_map)
print(f"pretokens: {len(freq_map)} unique, {sum(freq_map.values())} total")
print(f"distinct adjacent pairs: {len(pair_counts)}\n")

print("top 10 pairs by count:")
for pair, c in sorted(pair_counts.items(), key=lambda kv: (-kv[1], kv[0]))[:10]:
    print(f"  ({show(pair[0])}, {show(pair[1])}) -> {c}")

best_pair = get_best_pair(pair_counts)
new_id = 256
print(f"\nbest pair: ({show(best_pair[0])}, {show(best_pair[1])}) "
      f"= {best_pair} with count {pair_counts[best_pair]}")
print(f"merging into new id {new_id}\n")

merged = apply_merge(freq_map, best_pair, new_id)

print("pretokens that changed:")
for old_tuple, count in freq_map.items():
    new_tuple = next(k for k in apply_merge({old_tuple: count}, best_pair, new_id))
    if new_tuple != old_tuple:
        word = bytes(old_tuple).decode("utf-8")
        print(f"  '{word}' (x{count}): {old_tuple} -> {new_tuple}")

print(f"\ntotal symbols before: {sum(len(k) * v for k, v in freq_map.items())}")
print(f"total symbols after:  {sum(len(k) * v for k, v in merged.items())}")

# The merged pair should now be gone from the pair counts.
after_counts = count_pairs(merged)
print(f"\nbest pair still present after merge? {best_pair in after_counts}")
next_pair = get_best_pair(after_counts)
print(f"next best pair would be: ({show(next_pair[0])}, {show(next_pair[1])}) "
      f"= {next_pair} with count {after_counts[next_pair]}")


pretokens: 23 unique, 26 total
distinct adjacent pairs: 55

top 10 pairs by count:
  ('r', 'e') -> 5
  (' ', 'h') -> 4
  (' ', 't') -> 4
  ('h', 'e') -> 4
  ("'", 's') -> 3
  ('e', 'r') -> 3
  ('t', 'h') -> 3
  (' ', 'i') -> 2
  (' ', 's') -> 2
  (' ', 'w') -> 2

best pair: ('r', 'e') = (114, 101) with count 5
merging into new id 256

pretokens that changed:
  'here' (x1): (104, 101, 114, 101) -> (104, 101, 256)
  ' here' (x2): (32, 104, 101, 114, 101) -> (32, 104, 101, 256)
  ' pretokenized' (x1): (32, 112, 114, 101, 116, 111, 107, 101, 110, 105, 122, 101, 100) -> (32, 112, 256, 116, 111, 107, 101, 110, 105, 122, 101, 100)
  ' regex' (x1): (32, 114, 101, 103, 101, 120) -> (32, 256, 103, 101, 120)

total symbols before: 110
total symbols after:  105

best pair still present after merge? False
next best pair would be: ('h', 'e') = (104, 101) with count 4


### Train BPE -- calls and test

In [16]:
# Train on the same long string, with a small vocab so every merge is readable.
text = "here's here's here's some text and I'd like to see how it is getting pretokenized with the regex that we have!"
freq_map = pretokenize(text)

vocab, merges, special_token_ids = train_bpe(freq_map, vocab_size=270, special_tokens=[])

def render(tok_bytes):
    """Show a token's bytes as text, with the leading space marked as \u2581."""
    return tok_bytes.decode("utf-8", errors="replace").replace(" ", "\u2581")

print("\nmerges in order:")
for rank, (a, b) in enumerate(merges):
    new_id = 256 + rank
    print(f"  {rank:2}: ({render(vocab[a])!r}, {render(vocab[b])!r}) -> {new_id} = {render(vocab[new_id])!r}")

print("\nlearned tokens (ids >= 256):")
print("  " + "  ".join(render(vocab[i]) for i in range(256, len(vocab))))


Merge 0/14: (114, 101) -> 256  (count=5, token='b're'')
Training completed.
Vocab size: 270
Merges learned: 14

merges in order:
   0: ('r', 'e') -> 256 = 're'
   1: ('h', 'e') -> 257 = 'he'
   2: ('▁', 't') -> 258 = '▁t'
   3: ('he', 're') -> 259 = 'here'
   4: ("'", 's') -> 260 = "'s"
   5: ('k', 'e') -> 261 = 'ke'
   6: ('i', 't') -> 262 = 'it'
   7: ('h', 'a') -> 263 = 'ha'
   8: ('g', 'e') -> 264 = 'ge'
   9: ('▁', 'here') -> 265 = '▁here'
  10: ('▁', 'w') -> 266 = '▁w'
  11: ('▁', 's') -> 267 = '▁s'
  12: ('▁s', 'o') -> 268 = '▁so'
  13: ('▁so', 'm') -> 269 = '▁som'

learned tokens (ids >= 256):
  re  he  ▁t  here  's  ke  it  ha  ge  ▁here  ▁w  ▁s  ▁so  ▁som


In [22]:
# Train on a small real corpus. <|endoftext|> is held out as a special token.
DATA_PATH = "ml_data/TinyStoriesV2-GPT4_small.txt"
SPECIAL_TOKENS = ["<|endoftext|>"]

file_freq_map = pretokenize_file(DATA_PATH, SPECIAL_TOKENS)
print(f"pretokens: {len(file_freq_map)} unique, {sum(file_freq_map.values())} total")
print(f"bytes in corpus: {sum(len(k) * v for k, v in file_freq_map.items())}\n")

file_vocab, file_merges, file_special_ids = train_bpe(
    file_freq_map, vocab_size=400, special_tokens=SPECIAL_TOKENS
)

# Learned ids start after the 256 bytes plus the special tokens.
first_learned_id = 256 + len(SPECIAL_TOKENS)
print(f"\nspecial token ids: {file_special_ids}")

print("\nfirst 30 merges:")
for rank in range(min(30, len(file_merges))):
    print(f"  {rank:2}: {render(file_vocab[first_learned_id + rank])!r}")

print("\nlongest learned tokens:")
learned = [file_vocab[i] for i in range(first_learned_id, len(file_vocab))]
for tok in sorted(learned, key=len, reverse=True)[:15]:
    print(f"  {render(tok)!r}  ({len(tok)} bytes)")


pretokens: 252 unique, 744 total
bytes in corpus: 3009

Merge 0/143: (32, 116) -> 257  (count=94, token='b' t'')
Merge 50/143: (111, 116) -> 307  (count=9, token='b'ot'')
Merge 100/143: (356, 272) -> 357  (count=5, token='b' tower'')
Training completed.
Vocab size: 400
Merges learned: 143

special token ids: {'<|endoftext|>': 256}

first 30 merges:
   0: '▁t'
   1: 'he'
   2: '▁a'
   3: '▁s'
   4: '▁the'
   5: '▁w'
   6: 'ed'
   7: 'nd'
   8: '▁to'
   9: '▁b'
  10: '▁and'
  11: '▁T'
  12: 'ar'
  13: '▁h'
  14: 'om'
  15: 'er'
  16: '▁f'
  17: 'ou'
  18: 'it'
  19: 'in'
  20: '▁p'
  21: '▁c'
  22: '▁wa'
  23: '▁l'
  24: '▁The'
  25: 'll'
  26: 'id'
  27: '▁They'
  28: 'or'
  29: '▁hi'

longest learned tokens:
  '▁blocks'  (7 bytes)
  '▁scared'  (7 bytes)
  '▁wanted'  (7 bytes)
  '▁looked'  (7 bytes)
  '▁their'  (6 bytes)
  '▁tower'  (6 bytes)
  '▁liked'  (6 bytes)
  '▁They'  (5 bytes)
  '▁were'  (5 bytes)
  'locks'  (5 bytes)
  '▁ball'  (5 bytes)
  '▁with'  (5 bytes)
  '▁scar'  (5 bytes

In [28]:
# Encode with the vocab trained on TinyStories above.
tests = [
    "Once upon a time",
    "The little girl liked to play.",
    "<|endoftext|>Tom was scared.",
    "zqx",                      # unseen chars, should stay raw bytes
    "café",                     # multi-byte utf-8
]

for s in tests:
    ids = encode(s, file_merges, file_special_ids)
    # Round-trip check: decode(encode(s)) must give back the original.
    roundtrip = decode(ids, file_vocab)
    print(f"{s!r}")
    print(f"  ids ({len(ids)} vs {len(s.encode('utf-8'))} bytes): {ids}")
    print(f"  tokens: {[render(file_vocab[i]) for i in ids]}")
    print(f"  round-trip ok: {roundtrip == s}\n")


'Once upon a time'
  ids (13 vs 16 bytes): [79, 110, 387, 32, 117, 112, 111, 110, 259, 257, 105, 109, 101]
  tokens: ['O', 'n', 'ce', '▁', 'u', 'p', 'o', 'n', '▁a', '▁t', 'i', 'm', 'e']
  round-trip ok: True

'The little girl liked to play.'
  ids (15 vs 30 bytes): [84, 258, 280, 275, 116, 335, 350, 315, 108, 392, 265, 277, 108, 363, 46]
  tokens: ['T', 'he', '▁l', 'it', 't', 'le', '▁g', 'ir', 'l', '▁liked', '▁to', '▁p', 'l', 'ay', '.']
  round-trip ok: True

'<|endoftext|>Tom was scared.'
  ids (5 vs 28 bytes): [256, 388, 289, 351, 46]
  tokens: ['<|endoftext|>', 'Tom', '▁was', '▁scared', '.']
  round-trip ok: True

'zqx'
  ids (3 vs 3 bytes): [122, 113, 120]
  tokens: ['z', 'q', 'x']
  round-trip ok: True

'café'
  ids (5 vs 5 bytes): [99, 97, 102, 195, 169]
  tokens: ['c', 'a', 'f', '�', '�']
  round-trip ok: True



## Extras and checks

In [9]:
# Understanding unicode as we use utf-8 encodings for strings as we work through tokenization.
s = "hello 5 % ^ éşß"
s_enc = s.encode("utf-8")
s_enc_ints = list(s_enc)

print(s)
print(s_enc)
print(s_enc_ints)
print(len(s))
print(len(s_enc_ints))

hello 5 % ^ éşß
b'hello 5 % ^ \xc3\xa9\xc5\x9f\xc3\x9f'
[104, 101, 108, 108, 111, 32, 53, 32, 37, 32, 94, 32, 195, 169, 197, 159, 195, 159]
15
18
